### EV Charging Network — Dynamic Tariff Optimization
#### Notebook 1: Data Preprocessing & Feature Engineering

**Datasets Used:**
- **ACN-Data** (Caltech/JPL): Real EV session logs — 30K+ sessions, timestamps, kWh delivered, station/user IDs
- **ST-EVCDP** (Shenzhen UrbanEV): 247 charging stations, 5-min interval data — occupancy, volume, duration, price, spatial metadata

**Goal:** Align both datasets into a unified analytical base with engineered features ready for EDA and ML modelling.

**Assumptions & Limitations:**
- ACN timestamps are in America/Los_Angeles timezone; converted to UTC for uniformity
- ST-EVCDP price values are in CNY/kWh; treated as relative price index for tariff modelling
- Missing values in ST-EVCDP matrices: forward-filled (short gaps) then dropped if >10% missing per station
- ACN sessions with kWhDelivered = 0 are treated as failed/cancelled sessions and excluded

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 5)

# Get the project root (one level up from notebooks/)
BASE = os.path.dirname(os.path.abspath("__file__"))
BASE = os.path.join(BASE, "..")
BASE = os.path.normpath(BASE)

RAW   = os.path.join(BASE, "data", "raw") + "/"
PROC  = os.path.join(BASE, "data", "processed") + "/"
PLOTS = os.path.join(BASE, "plots") + "/"
OUT   = os.path.join(BASE, "outputs") + "/"

for d in [PROC, PLOTS, OUT]:
    os.makedirs(d, exist_ok=True)

print(f"BASE  : {BASE}")
print(f"RAW   : {RAW}")
print(f"PROC  : {PROC}")
print(f"PLOTS : {PLOTS}")
print(f"OUT   : {OUT}")
print(" Libraries loaded | Directories ready")

BASE  : c:\Users\hp\OneDrive\Desktop\EV_Tariff_Optimization
RAW   : c:\Users\hp\OneDrive\Desktop\EV_Tariff_Optimization\data\raw/
PROC  : c:\Users\hp\OneDrive\Desktop\EV_Tariff_Optimization\data\processed/
PLOTS : c:\Users\hp\OneDrive\Desktop\EV_Tariff_Optimization\plots/
OUT   : c:\Users\hp\OneDrive\Desktop\EV_Tariff_Optimization\outputs/
 Libraries loaded | Directories ready


### 1.1 Load ACN-Data (Caltech/JPL Sessions)

In [2]:
acn_raw = pd.read_excel(RAW + "acndata_sessions.json.xlsx", sheet_name="Sheet1")
print(f"ACN Raw shape : {acn_raw.shape}")
print(f"Columns       : {acn_raw.columns.tolist()}")
acn_raw.head(3)

ACN Raw shape : (16304, 27)
Columns       : ['_meta', 'end', 'min_kWh', 'site', 'start', '_items', '_id', 'clusterID', 'connectionTime', 'disconnectTime', 'doneChargingTime', 'kWhDelivered', 'sessionID', 'siteID', 'spaceID', 'stationID', 'timezone', 'userID', 'userInputs', 'WhPerMile', 'kWhRequested', 'milesRequested', 'minutesAvailable', 'modifiedAt', 'paymentRequired', 'requestedDeparture', 'userID.1']


,_meta,end,min_kWh,site,start,_items,_id,clusterID,connectionTime,disconnectTime,...,userID,userInputs,WhPerMile,kWhRequested,milesRequested,minutesAvailable,modifiedAt,paymentRequired,requestedDeparture,userID.1
0,NaN,NaN,NaN,caltech,NaN,NaN,5bc90cb9f9af8b0d7fe77cd2,39.0,"Wed, 25 Apr 2018 11:08:04 GMT","Wed, 25 Apr 2018 13:20:10 GMT",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,5bc90cb9f9af8b0d7fe77cd3,39.0,"Wed, 25 Apr 2018 13:45:10 GMT","Thu, 26 Apr 2018 00:56:16 GMT",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,5bc90cb9f9af8b0d7fe77cd4,39.0,"Wed, 25 Apr 2018 13:45:50 GMT","Wed, 25 Apr 2018 23:04:45 GMT",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
acn = acn_raw[[
    '_id', 'siteID', 'stationID', 'spaceID', 'clusterID',
    'connectionTime', 'disconnectTime', 'doneChargingTime',
    'kWhDelivered', 'userID', 'timezone'
]].copy()

acn.columns = [
    'session_id', 'site_id', 'station_id', 'space_id', 'cluster_id',
    'connection_time', 'disconnect_time', 'done_charging_time',
    'kwh_delivered', 'user_id', 'timezone'
]

for col in ['connection_time', 'disconnect_time', 'done_charging_time']:
    acn[col] = pd.to_datetime(acn[col], utc=True, errors='coerce')

acn = acn.dropna(subset=['connection_time', 'disconnect_time'])
acn = acn[acn['kwh_delivered'] > 0].copy()

acn['session_duration_hr']  = (acn['disconnect_time'] - acn['connection_time']).dt.total_seconds() / 3600
acn['charging_duration_hr'] = (acn['done_charging_time'] - acn['connection_time']).dt.total_seconds() / 3600
acn['idle_time_hr']         = (acn['disconnect_time'] - acn['done_charging_time']).dt.total_seconds() / 3600
acn['avg_power_kw']         = acn['kwh_delivered'] / acn['session_duration_hr'].replace(0, np.nan)

acn['hour']        = acn['connection_time'].dt.hour
acn['day_of_week'] = acn['connection_time'].dt.dayofweek
acn['is_weekend']  = acn['day_of_week'].isin([5, 6]).astype(int)
acn['month']       = acn['connection_time'].dt.month
acn['date']        = acn['connection_time'].dt.date

FIXED_TARIFF         = 15.0
acn['revenue_fixed'] = acn['kwh_delivered'] * FIXED_TARIFF

acn = acn[(acn['session_duration_hr'] > 0) & (acn['session_duration_hr'] < 24)]

print(f" ACN cleaned shape : {acn.shape}")
print(f"   Date range        : {acn['connection_time'].min()} → {acn['connection_time'].max()}")
print(f"   Sites             : {acn['site_id'].unique()}")
print(f"   Total kWh         : {acn['kwh_delivered'].sum():,.0f}")
acn.head(3)

 ACN cleaned shape : (14848, 21)
   Date range        : 2018-04-25 11:08:04+00:00 → 2018-12-16 03:01:46+00:00
   Sites             : [2.]
   Total kWh         : 132,782


,session_id,site_id,station_id,space_id,cluster_id,connection_time,disconnect_time,done_charging_time,kwh_delivered,user_id,...,session_duration_hr,charging_duration_hr,idle_time_hr,avg_power_kw,hour,day_of_week,is_weekend,month,date,revenue_fixed
0,5bc90cb9f9af8b0d7fe77cd2,2.0,2-39-78-362,CA-496,39.0,2018-04-25 11:08:04+00:00,2018-04-25 13:20:10+00:00,2018-04-25 13:21:10+00:00,7.932,NaN,...,2.201667,2.218333,-0.016667,3.602725,11,2,0,4,2018-04-25,118.980
1,5bc90cb9f9af8b0d7fe77cd3,2.0,2-39-95-27,CA-319,39.0,2018-04-25 13:45:10+00:00,2018-04-26 00:56:16+00:00,2018-04-25 16:44:15+00:00,10.013,NaN,...,11.185000,2.984722,8.200278,0.895217,13,2,0,4,2018-04-25,150.195
2,5bc90cb9f9af8b0d7fe77cd4,2.0,2-39-79-380,CA-489,39.0,2018-04-25 13:45:50+00:00,2018-04-25 23:04:45+00:00,2018-04-25 14:51:44+00:00,5.257,NaN,...,9.315278,1.098333,8.216944,0.564342,13,2,0,4,2018-04-25,78.855


###  1.2 Load ST-EVCDP (Shenzhen UrbanEV) — Time-Series Matrices

In [4]:
occ_raw     = pd.read_csv(RAW + "occupancy.csv")
vol_raw     = pd.read_csv(RAW + "volume.csv")
dur_raw     = pd.read_csv(RAW + "duration.csv")
price_raw   = pd.read_csv(RAW + "price.csv")
time_df     = pd.read_csv(RAW + "time.csv")
info_df     = pd.read_csv(RAW + "information.csv")
stations_df = pd.read_csv(RAW + "stations.csv")
adj_df      = pd.read_csv(RAW + "adj.csv",      index_col='node_id')
dist_df     = pd.read_csv(RAW + "distance.csv", index_col='Unnamed: 0')

print(f"Occupancy  : {occ_raw.shape}")
print(f"Volume     : {vol_raw.shape}")
print(f"Duration   : {dur_raw.shape}")
print(f"Price      : {price_raw.shape}")
print(f"Time index : {time_df.shape}")
print(f"Info       : {info_df.shape}")
print(f"Stations   : {stations_df.shape}")

Occupancy  : (8640, 248)
Volume     : (8640, 248)
Duration   : (8640, 248)
Price      : (8640, 248)
Time index : (8640, 6)
Info       : (247, 10)
Stations   : (1706, 6)


In [5]:
time_df['datetime'] = pd.to_datetime(
    time_df[['year','month','day','hour','minute','second']]
)
datetime_index = time_df['datetime']

print(f"Time range  : {datetime_index.iloc[0]} → {datetime_index.iloc[-1]}")
print(f"Interval    : {(datetime_index.iloc[1]-datetime_index.iloc[0]).total_seconds()/60:.0f} minutes")
print(f"Total steps : {len(datetime_index)}")

Time range  : 2022-06-19 00:00:00 → 2022-07-18 23:55:00
Interval    : 5 minutes
Total steps : 8640


In [6]:
def attach_datetime(df, datetime_index):
    df = df.copy()
    df.insert(0, 'datetime', datetime_index.values)
    df = df.drop(columns=['timestamp'], errors='ignore')
    df = df.set_index('datetime')
    df.columns = df.columns.astype(str)
    return df

occ   = attach_datetime(occ_raw,   datetime_index)
vol   = attach_datetime(vol_raw,   datetime_index)
dur   = attach_datetime(dur_raw,   datetime_index)
price = attach_datetime(price_raw, datetime_index)

print(" Datetime index attached to all matrices")
print(f"Occupancy range : {occ.min().min():.1f} – {occ.max().max():.1f}")
print(f"Price range     : {price.min().min():.3f} – {price.max().max():.3f}")

 Datetime index attached to all matrices
Occupancy range : 0.0 – 220.0
Price range     : 0.250 – 1.470


In [7]:
missing_pct   = occ.isnull().mean() * 100
good_stations = missing_pct[missing_pct <= 10].index.tolist()

occ   = occ[good_stations]
vol   = vol[[c for c in good_stations if c in vol.columns]]
dur   = dur[[c for c in good_stations if c in dur.columns]]
price = price[[c for c in good_stations if c in price.columns]]

occ   = occ.ffill().bfill()
vol   = vol.ffill().bfill()
dur   = dur.ffill().bfill()
price = price.ffill().bfill()

print(f" Retained {len(good_stations)} stations after quality filter")
print(f"   Stations removed due to >10% missing : {247 - len(good_stations)}")

 Retained 247 stations after quality filter
   Stations removed due to >10% missing : 0


### 1.3 Feature Engineering — Derived Metrics

**Features engineered:**
- **Utilization Rate** = occupied chargers / total chargers (clipped 0–1)
- **Congestion Flag** = utilization > 0.80
- **Off-Peak Flag** = utilization < 0.30
- **Revenue Proxy** = volume × price × (5/60) per interval

In [8]:
info_df['grid'] = info_df['grid'].astype(str)
station_meta = info_df[['grid','count','fast_count','slow_count',
                         'area','lon','la','CBD','dynamic_pricing']].copy()
station_meta = station_meta.rename(columns={'grid':'station_id','la':'lat'})
station_meta = station_meta.set_index('station_id')

util_df = occ.copy()
for col in util_df.columns:
    if col in station_meta.index:
        total = station_meta.loc[col, 'count']
        if total > 0:
            util_df[col] = (util_df[col] / total).clip(0, 1)

congestion_df = (util_df > 0.8).astype(int)
offpeak_df    = (util_df < 0.3).astype(int)
revenue_df    = vol * price * (5/60)

print("Engineered features created")
print(f"   Mean utilization rate : {util_df.mean().mean():.2%}")
print(f"   Congestion %          : {congestion_df.mean().mean():.2%}")
print(f"   Off-peak %            : {offpeak_df.mean().mean():.2%}")

Engineered features created
   Mean utilization rate : 28.02%
   Congestion %          : 1.02%
   Off-peak %            : 60.85%


In [9]:
SAMPLE_STATIONS = good_stations[:50]

panel_frames = []
for st in SAMPLE_STATIONS:
    if st not in util_df.columns:
        continue
    df_st = pd.DataFrame({
        'datetime'    : util_df.index,
        'station_id'  : st,
        'utilization' : util_df[st].values,
        'volume'      : vol[st].values        if st in vol.columns        else np.nan,
        'duration'    : dur[st].values        if st in dur.columns        else np.nan,
        'price'       : price[st].values      if st in price.columns      else np.nan,
        'congestion'  : congestion_df[st].values,
        'offpeak'     : offpeak_df[st].values,
        'revenue'     : revenue_df[st].values if st in revenue_df.columns else np.nan,
    })
    panel_frames.append(df_st)

panel = pd.concat(panel_frames, ignore_index=True)

panel['hour']        = panel['datetime'].dt.hour
panel['day_of_week'] = panel['datetime'].dt.dayofweek
panel['is_weekend']  = panel['day_of_week'].isin([5,6]).astype(int)
panel['month']       = panel['datetime'].dt.month
panel['time_slot']   = pd.cut(
    panel['hour'],
    bins=[-1, 6, 9, 17, 20, 24],
    labels=['Night','AM_Peak','Daytime','PM_Peak','Evening']
)

panel = panel.merge(
    station_meta.reset_index(),
    left_on='station_id', right_on='station_id', how='left'
)

print(f" Panel dataset shape : {panel.shape}")
panel.head(3)

 Panel dataset shape : (432000, 22)


,datetime,station_id,utilization,volume,duration,price,congestion,offpeak,revenue,hour,...,month,time_slot,count,fast_count,slow_count,area,lon,lat,CBD,dynamic_pricing
0,2022-06-19 00:00:00,102,0.4,2.858333,0.49,0.924,0,0,0.220092,0,...,6,Night,30,3,27,0.71,114.103,22.54041,0,0
1,2022-06-19 00:05:00,102,0.4,4.375000,0.75,0.924,0,0,0.336875,0,...,6,Night,30,3,27,0.71,114.103,22.54041,0,0
2,2022-06-19 00:10:00,102,0.4,4.375000,0.75,0.924,0,0,0.336875,0,...,6,Night,30,3,27,0.71,114.103,22.54041,0,0


In [10]:
acn.to_csv(PROC + "acn_processed.csv",        index=False)
panel.to_csv(PROC + "urbanev_panel.csv",       index=False)
util_df.to_csv(PROC + "utilization_matrix.csv")
price.to_csv(PROC + "price_matrix.csv")
station_meta.to_csv(PROC + "station_meta.csv")

print(" All processed files saved to data/processed/")
print(f"   acn_processed.csv       → {acn.shape[0]:,} sessions")
print(f"   urbanev_panel.csv       → {panel.shape[0]:,} rows")
print(f"   utilization_matrix.csv  → {util_df.shape}")
print(f"   price_matrix.csv        → {price.shape}")
print(f"   station_meta.csv        → {station_meta.shape}")

 All processed files saved to data/processed/
   acn_processed.csv       → 14,848 sessions
   urbanev_panel.csv       → 432,000 rows
   utilization_matrix.csv  → (8640, 247)
   price_matrix.csv        → (8640, 247)
   station_meta.csv        → (247, 8)


In [11]:
summary = pd.DataFrame({
    'Dataset'         : ['ACN-Data', 'ST-EVCDP'],
    'Records'         : [acn.shape[0], panel.shape[0]],
    'Stations'        : [acn['station_id'].nunique(), len(good_stations)],
    'Date_Start'      : [str(acn['connection_time'].min().date()), '2022-06-19'],
    'Date_End'        : [str(acn['connection_time'].max().date()), '2022-08-17'],
    'Key_Features'    : [
        'kWh, session_duration, idle_time, avg_power, revenue_fixed',
        'utilization, volume, price, congestion, offpeak, revenue_proxy'
    ],
    'Missing_Handled' : [
        'Dropped 0-kWh and invalid duration sessions',
        'Forward-fill + station quality filter (>10% missing dropped)'
    ]
})
summary.to_csv(OUT + "data_summary.csv", index=False)
print(summary.to_string())
print("\n Notebook 1 complete ,Run Notebook 2 next.")

    Dataset  Records  Stations  Date_Start    Date_End                                                    Key_Features                                               Missing_Handled
0  ACN-Data    14848        54  2018-04-25  2018-12-16      kWh, session_duration, idle_time, avg_power, revenue_fixed                   Dropped 0-kWh and invalid duration sessions
1  ST-EVCDP   432000       247  2022-06-19  2022-08-17  utilization, volume, price, congestion, offpeak, revenue_proxy  Forward-fill + station quality filter (>10% missing dropped)

 Notebook 1 complete ,Run Notebook 2 next.
